<a href="https://colab.research.google.com/github/emanfatimaa05/urdu-ocr-codesaviours-si26-eman/blob/main/week4_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!git clone https://github.com/emanfatimaa05/urdu-ocr-codesaviours-si26-eman.git
%cd urdu-ocr-codesaviours-si26-eman

Cloning into 'urdu-ocr-codesaviours-si26-eman'...
remote: Enumerating objects: 409, done.
remote: Counting objects: 100% (409/409), done.
remote: Compressing objects: 100% (398/398), done.
remote: Total 409 (delta 44), reused 218 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (409/409), 4.66 MiB | 35.33 MiB/s, done.
Resolving deltas: 100% (44/44), done.
/content/urdu-ocr-codesaviours-si26-eman


In [10]:
!pip install transformers torch pillow pandas sentencepiece -q

In [11]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, ViTImageProcessor, RobertaTokenizer
from torch.optim import AdamW
from PIL import Image
import pandas as pd

In [12]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cpu':
    print('WARNING: No GPU detected.')
    print('Go to Runtime > Change runtime type > GPU')

image_processor = ViTImageProcessor.from_pretrained('microsoft/trocr-base-printed')
tokenizer = RobertaTokenizer.from_pretrained('microsoft/trocr-base-printed')
processor = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)

model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-printed')
model = model.to(device)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

print('Model loaded successfully!')
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

Using device: cuda


Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-printed
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully!
Model parameters: 333,921,792


In [13]:
class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, processor):
        self.data = pd.read_csv(csv_path)
        self.processor = processor
        print(f'Dataset loaded: {len(self.data)} samples')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image = Image.open(row['image']).convert('RGB')
        encoding = self.processor(image, return_tensors='pt')
        pixel_values = encoding.pixel_values.squeeze()
        labels = self.processor.tokenizer(
            row['text'],
            padding='max_length',
            max_length=128,
            truncation=True
        ).input_ids
        labels = torch.tensor(labels)
        return {'pixel_values': pixel_values, 'labels': labels}

In [14]:
dataset = UrduOCRDataset('data/labels.csv', processor)

torch.manual_seed(42)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, test_size]
)
print(f'Training samples: {train_size}')
print(f'Testing samples: {test_size}')

Dataset loaded: 205 samples
Training samples: 164
Testing samples: 41


In [15]:
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4)

optimizer = AdamW(model.parameters(), lr=5e-5)

print(f'Training batches per epoch: {len(train_loader)}')
print('Ready to train!')

Training batches per epoch: 41
Ready to train!


In [18]:
num_epochs = 15
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    print(f'\nEpoch {epoch + 1}/{num_epochs}')
    print('-' * 30)
    for batch_idx, batch in enumerate(train_loader):
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        if batch_idx % 10 == 0:
            print(f'  Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}')

    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch + 1} complete | Average Loss: {avg_loss:.4f}')

print('\nTraining complete!')


Epoch 1/15
------------------------------
  Batch 0/41 | Loss: 3.4231
  Batch 10/41 | Loss: 3.4400
  Batch 20/41 | Loss: 3.3658
  Batch 30/41 | Loss: 2.3808
  Batch 40/41 | Loss: 2.3603
Epoch 1 complete | Average Loss: 2.7997

Epoch 2/15
------------------------------
  Batch 0/41 | Loss: 2.9489
  Batch 10/41 | Loss: 2.5835
  Batch 20/41 | Loss: 2.8685
  Batch 30/41 | Loss: 2.5257
  Batch 40/41 | Loss: 3.0325
Epoch 2 complete | Average Loss: 2.7834

Epoch 3/15
------------------------------
  Batch 0/41 | Loss: 2.4759
  Batch 10/41 | Loss: 2.5181
  Batch 20/41 | Loss: 2.4573
  Batch 30/41 | Loss: 3.0270
  Batch 40/41 | Loss: 2.3050
Epoch 3 complete | Average Loss: 2.7448

Epoch 4/15
------------------------------
  Batch 0/41 | Loss: 2.7922
  Batch 10/41 | Loss: 2.0095
  Batch 20/41 | Loss: 2.7423
  Batch 30/41 | Loss: 2.5047
  Batch 40/41 | Loss: 2.9887
Epoch 4 complete | Average Loss: 2.7289

Epoch 5/15
------------------------------
  Batch 0/41 | Loss: 1.8774
  Batch 10/41 | Loss:

In [19]:
model.eval()
print('=== Model Evaluation on Test Images ===')
print()

correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels']

        generated_ids = model.generate(pixel_values)
        generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)
        actual_text = processor.batch_decode(labels, skip_special_tokens=True)

        for pred, actual in zip(generated_text, actual_text):
            total += 1
            if pred.strip() == actual.strip():
                correct += 1
            print(f'Predicted: {pred}')
            print(f'Actual: {actual}')
            print()

accuracy = (correct / total) * 100 if total > 0 else 0
print(f'Accuracy: {accuracy:.1f}% ({correct}/{total} correct)')

=== Model Evaluation on Test Images ===



/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1625: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Predicted: ��������������������
Actual: عظمیٰ بخاری نے کہا کہ 'پیپلز پارٹی کی پنجاب والی قیادت کا بارش میں زیادہ پنجاب حکومت پر فوکس ہے، 

Predicted: 
Actual: کراچی پاکستان کا سب سے بڑا شہر ہے

Predicted: 
Actual: محنت کامیابی کی کنجی ہے

Predicted: 
Actual: اسسٹنٹ ڈائریکٹر ایگریکلچر آن فارم واٹر مینجمنٹ

Predicted: 
Actual: نقیب نے مجھے طلب کیا میرے اثاثوں میں... ...صرف تم نکلے۔

Predicted: 
Actual: دیکھ کر جائے اس سڑک کے سگنلز تم نے دل کا سلام آباد بنا ڈالا

Predicted: 
Actual: جس دن تمہیں قرآن سمجھ میں آگیا اُس دن تمہارا نہ کوئی مسلک ہو گا نہ فرقہ

Predicted: 
Actual: مخالفت کے باوجود برصغیر میں ملتِ اسلامیہ دینِ اسلام پر اور اسی نظریے

Predicted: 
Actual: اسٹائل آپ کے زمانے کا اور ریٹ آپ کے اباجی کے زمانے کا

Predicted: 
Actual: عناصر جو اس مملکت کے بنیادی نظریے پر ایمان نہیں رکھتے تھے۔ وہ

Predicted: 
Actual: ہوٹل کا عملہ نہایت دوستانہ ہے۔ عملہ نے

Predicted: ��������������������
Actual: اے اللہ! ہمیں مانگنے کا سلیقہ نہیں آتا، پھر بھی ہر روز ہاتھ پھیلائے اس اُمید کے ساتھ تجھ سے ما